#Silver Layer

##Data Quality checks on bronze layer

In [0]:
SELECT 
  COUNT(*) as row_count,
  COUNT(DISTINCT `Lärosäte`) as unique_schools,
  COUNT(DISTINCT `Tidsperiod`) as unique_years,
  COUNT(DISTINCT `Kön`) as unique_gender
FROM bronze_education.university_sun;


In [0]:
-- Check nulls per column
SELECT 
  COUNT_IF(`Lärosäte` IS NULL) as null_larosate,
  COUNT_IF(`Examen` IS NULL) as null_examen,
  COUNT_IF(`Huvudinriktning[0]` IS NULL) as null_inriktning,
  COUNT_IF(`Värde` IS NULL) as null_varde,
  COUNT_IF(`_rescued_data` IS NULL) as null_rescue
FROM bronze_education.university_sun;

In [0]:
SELECT 
  'Tidsperiod' as column_name,
  'STRING' as data_type,
  COUNT(*) as total_rows,
  COUNT_IF(`Tidsperiod` IS NULL) as null_count,
  ROUND(COUNT_IF(`Tidsperiod` IS NULL) / COUNT(*) * 100, 2) as null_pct,
  COUNT(DISTINCT `Tidsperiod`) as distinct_count
FROM bronze_education.university_sun

UNION ALL

SELECT 'Lärosäte', 'STRING', COUNT(*), COUNT_IF(`Lärosäte` IS NULL), ROUND(COUNT_IF(`Lärosäte` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `Lärosäte`) FROM bronze_education.university_sun

UNION ALL

SELECT 'Huvudinriktning', 'STRING', COUNT(*), COUNT_IF(`Huvudinriktning[0]` IS NULL), ROUND(COUNT_IF(`Huvudinriktning[0]` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `Huvudinriktning[0]`) FROM bronze_education.university_sun

UNION ALL

SELECT 'Huvudomradesgrupp', 'STRING', COUNT(*), COUNT_IF(`Huvudområdesgrupp[1]` IS NULL), ROUND(COUNT_IF(`Huvudområdesgrupp[1]` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `Huvudområdesgrupp[1]`) FROM bronze_education.university_sun

UNION ALL

SELECT 'Varde', 'NUMERIC', COUNT(*), COUNT_IF(`Värde` IS NULL), ROUND(COUNT_IF(`Värde` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `Värde`) FROM bronze_education.university_sun

UNION ALL

SELECT 'Kon', 'STRING', COUNT(*), COUNT_IF(`Kön` IS NULL), ROUND(COUNT_IF(`Kön` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `Kön`) FROM bronze_education.university_sun;

In [0]:
SELECT 
  CAST(REPLACE(`Värde`, ',', '') AS INT) as value,
  COUNT(*) as count
FROM bronze_education.university_sun
GROUP BY CAST(REPLACE(`Värde`, ',', '') AS INT)
ORDER BY value
LIMIT 20;

In [0]:
CREATE SCHEMA IF NOT EXISTS silver_education

In [0]:

Drop TABLE if exists silver_education.university_sun;

CREATE OR REPLACE TABLE silver_education.university_uka AS
SELECT 
  CAST(LEFT(REGEXP_REPLACE(`Tidsperiod`, '[^0-9/]', ''), 4) AS INT) as year,
  raw.`Lärosäte` as institution,
  `Huvudinriktning[0]` as subject_field,
  `Huvudområdesgrupp[1]` as subject_specialization,
  `Kön` as gender,
  TRY_CAST(ROUND(TRY_CAST(REPLACE(CAST(`Värde` AS STRING), ',', '.') AS FLOAT), 0) AS INT) as student_count,
  lookup.`Län` as region_name
FROM bronze_education.university_sun AS raw
LEFT JOIN bronze_education.university_lan_lookup AS lookup 
  ON raw.`Lärosäte` = lookup.`Lärosäte`;

In [0]:
DESCRIBE silver_education.university_uka;

In [0]:
DESCRIBE bronze_education.yh_region_subject

In [0]:
SELECT 
  'kon' as column_name,
  'STRING' as data_type,
  COUNT(*) as total_rows,
  COUNT_IF(`kön` IS NULL) as null_count,
  ROUND(COUNT_IF(`kön` IS NULL) / COUNT(*) * 100, 2) as null_pct,
  COUNT(DISTINCT `kön`) as distinct_count
FROM bronze_education.yh_region_subject

UNION ALL

SELECT 'utbildningens_inriktning', 'STRING', COUNT(*), COUNT_IF(`utbildningens_inriktning` IS NULL), ROUND(COUNT_IF(`utbildningens_inriktning` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `utbildningens_inriktning`) FROM bronze_education.yh_region_subject

UNION ALL

SELECT 'region_dar_utbildningen', 'STRING', COUNT(*), COUNT_IF(`region_där_utbildningen_bedrivs` IS NULL), ROUND(COUNT_IF(`region_där_utbildningen_bedrivs` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `region_där_utbildningen_bedrivs`) FROM bronze_education.yh_region_subject

UNION ALL

SELECT 'tabellinnehal', 'STRING', COUNT(*), COUNT_IF(`tabellinnehåll` IS NULL), ROUND(COUNT_IF(`tabellinnehåll` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `tabellinnehåll`) FROM bronze_education.yh_region_subject

UNION ALL

SELECT 'ar', 'STRING', COUNT(*), COUNT_IF(`år` IS NULL), ROUND(COUNT_IF(`år` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `år`) FROM bronze_education.yh_region_subject

UNION ALL

SELECT 'value', 'DOUBLE', COUNT(*), COUNT_IF(`value` IS NULL), ROUND(COUNT_IF(`value` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `value`) FROM bronze_education.yh_region_subject;

In [0]:
SELECT 
  `value`,
  COUNT(*) as count
FROM bronze_education.yh_region_subject
WHERE `value` IS NULL OR `value` = 0 OR CAST(`value` AS INT) = 0
GROUP BY `value`
ORDER BY `value`;


In [0]:
CREATE OR REPLACE TABLE silver_education.yh_region_lookup (
  yh_region_name STRING,
  lan_standardized STRING
);

INSERT INTO silver_education.yh_region_lookup VALUES
('Stockholms län', 'Stockholm'),
('Uppsala län', 'Uppsala'),
('Södermanlands län', 'Södermanland'),
('Östergötlands län', 'Östergötland'),
('Jönköpings län', 'Jönköping'),
('Kronobergs län', 'Kronoberg'),
('Kalmar län', 'Kalmar'),
('Gotlands län', 'Gotland'),
('Blekinge län', 'Blekinge'),
('Skåne län', 'Skåne'),
('Hallands län', 'Halland'),
('Västra Götalands län', 'Västra Götaland'),
('Värmlands län', 'Värmland'),
('Örebro län', 'Örebro'),
('Västmanlands län', 'Västmanland'),
('Dalarnas län', 'Dalarna'),
('Gävleborgs län', 'Gävleborg'),
('Västernorrlands län', 'Västernorrland'),
('Jämtlands län', 'Jämtland'),
('Västerbottens län', 'Västerbotten'),
('Norrbottens län', 'Norrbotten'),
('Uppgift saknas', 'Unknown');

In [0]:
CREATE OR REPLACE TABLE silver_education.yh_region_subject AS
SELECT 
  CAST(`år` AS INT) as year,
  `kön` as gender,
  `utbildningens_inriktning` as subject_direction,
  COALESCE(lookup.lan_standardized, `region_där_utbildningen_bedrivs`) as region_name,
  CAST(`value` AS INT) as student_count,
  `value` IS NULL as program_not_offered
FROM bronze_education.yh_region_subject AS bronze
LEFT JOIN silver_education.yh_region_lookup AS lookup 
  ON bronze.`region_där_utbildningen_bedrivs` = lookup.yh_region_name;

In [0]:
MERGE INTO silver_education.yh_region_subject AS yh
USING silver_education.yh_region_lookup AS lookup
ON yh.region_name = lookup.yh_region_name
WHEN MATCHED THEN UPDATE SET region_name = lookup.lan_standardized;

In [0]:
DESCRIBE silver_education.yh_region_subject

In [0]:
SELECT 
  'year' as column_name,
  'INT' as data_type,
  COUNT(*) as total_rows,
  COUNT_IF(`year` IS NULL) as null_count,
  ROUND(COUNT_IF(`year` IS NULL) / COUNT(*) * 100, 2) as null_pct,
  COUNT(DISTINCT `year`) as distinct_count
FROM silver_education.yh_region_subject

UNION ALL

SELECT 'gender', 'STRING', COUNT(*), COUNT_IF(`gender` IS NULL), ROUND(COUNT_IF(`gender` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `gender`) FROM silver_education.yh_region_subject

UNION ALL

SELECT 'subject_direction', 'STRING', COUNT(*), COUNT_IF(`subject_direction` IS NULL), ROUND(COUNT_IF(`subject_direction` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `subject_direction`) FROM silver_education.yh_region_subject

UNION ALL

SELECT 'region_name', 'STRING', COUNT(*), COUNT_IF(`region_name` IS NULL), ROUND(COUNT_IF(`region_name` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `region_name`) FROM silver_education.yh_region_subject

UNION ALL

SELECT 'student_count', 'INT', COUNT(*), COUNT_IF(`student_count` IS NULL), ROUND(COUNT_IF(`student_count` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `student_count`) FROM silver_education.yh_region_subject

UNION ALL

SELECT 'program_not_offered', 'BOOLEAN', COUNT(*), COUNT_IF(`program_not_offered` IS NULL), ROUND(COUNT_IF(`program_not_offered` IS NULL) / COUNT(*) * 100, 2), COUNT(DISTINCT `program_not_offered`) FROM silver_education.yh_region_subject;

In [0]:
SELECT DISTINCT region_name from silver_education.yh_region_subject

In [0]:
SELECT * FROM silver_education.yh_region_lookup;

In [0]:
CREATE OR REPLACE TABLE silver_education.yh_region_subject AS
SELECT 
  CAST(bronze.`år` AS INT) as year,
  bronze.`kön` as gender,
  bronze.`utbildningens_inriktning` as subject_direction,
  COALESCE(lookup.lan_standardized, bronze.`region_där_utbildningen_bedrivs`) as region_name,
  CAST(bronze.`value` AS INT) as student_count,
  bronze.`value` IS NULL as program_not_offered
FROM bronze_education.yh_region_subject AS bronze
LEFT JOIN silver_education.yh_region_lookup AS lookup 
  ON bronze.`region_där_utbildningen_bedrivs` = lookup.yh_region_name;

In [0]:
SELECT 
  yh.region_name as yh_region,
  lookup.lan_standardized as standardized
FROM bronze_education.yh_region_subject yh
LEFT JOIN silver_education.yh_region_lookup lookup 
  ON yh.region_name = lookup.yh_region_name
WHERE lookup.lan_standardized IS NOT NULL
LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE silver_education.yh_region_subject AS
SELECT 
  CAST(`år` AS INT) as year,
  `kön` as gender,
  'YH' as institution,
  `utbildningens_inriktning` as subject_direction,
  CASE 
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Stockholms län' THEN 'Stockholm'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Uppsala län' THEN 'Uppsala'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Södermanlands län' THEN 'Södermanland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Östergötlands län' THEN 'Östergötland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Jönköpings län' THEN 'Jönköping'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Kronobergs län' THEN 'Kronoberg'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Kalmar län' THEN 'Kalmar'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Gotlands län' THEN 'Gotland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Blekinge län' THEN 'Blekinge'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Skåne län' THEN 'Skåne'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Hallands län' THEN 'Halland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Västra Götalands län' THEN 'Västra Götaland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Värmlands län' THEN 'Värmland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Örebro län' THEN 'Örebro'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Västmanlands län' THEN 'Västmanland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Dalarnas län' THEN 'Dalarna'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Gävleborgs län' THEN 'Gävleborg'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Västernorrlands län' THEN 'Västernorrland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Jämtlands län' THEN 'Jämtland'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Västerbottens län' THEN 'Västerbotten'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Norrbottens län' THEN 'Norrbotten'
    WHEN TRIM(`region_där_utbildningen_bedrivs`) = 'Uppgift saknas' THEN 'Unknown'
    ELSE TRIM(`region_där_utbildningen_bedrivs`)
  END as region_name,
  CAST(`value` AS INT) as student_count,
  `value` IS NULL as program_not_offered
FROM bronze_education.yh_region_subject;

In [0]:
SELECT * FROM silver_education.yh_region_subject

In [0]:
SELECT 
  `region_där_utbildningen_bedrivs`,
  LENGTH(`region_där_utbildningen_bedrivs`) as char_length,
  CONCAT('[', `region_där_utbildningen_bedrivs`, ']') as with_brackets
FROM bronze_education.yh_region_subject
GROUP BY `region_där_utbildningen_bedrivs`
ORDER BY 1;